In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess; subprocess.run(['pip', 'install', '-q', '--upgrade', 'openpyxl'])

import time
import requests
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY          = 'neom-9172-sjou-mzxw'
BASE_URL         = 'https://api.boxtobox.ai/v1/wy/api/v3'
COMPETITION_WYID = 364
SEASON_ID        = 191622
SLEEP_SECONDS    = 0.15
OUT_XLSX         = '/content/drive/MyDrive/Event data/Wyscout/epl.xlsx'

def api(path, **params):
    r = requests.get(f'{BASE_URL}{path}', params={'api_key': API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

def try_endpoint(path, **params):
    try:
        r = requests.get(f'{BASE_URL}{path}', params={'api_key': API_KEY, **params}, timeout=30)
        data = r.json()
        if isinstance(data, dict) and 'error' in data:
            print(f'  ✗ {path} → {data["error"]["code"]} {data["error"]["message"]}')
            return None
        print(f'  ✓ {path} → {r.status_code}, type={type(data).__name__}, keys={list(data.keys()) if isinstance(data, dict) else f"list[{len(data)}]"}')
        return data
    except Exception as e:
        print(f'  ✗ {path} → {e}')
        return None

# ── Probe available endpoints ─────────────────────────────────────────────────
print('Probing endpoints...')
candidates = [
    (f'/competitions/{COMPETITION_WYID}/matches',       {'seasonId': SEASON_ID}),
    (f'/seasons/{SEASON_ID}/matches',                   {}),
    (f'/competitions/{COMPETITION_WYID}/seasons/{SEASON_ID}/matches', {}),
    ('/matches',                                        {'compId': COMPETITION_WYID, 'seasonId': SEASON_ID}),
    (f'/seasons/{SEASON_ID}/matches/rounds',            {}),
    (f'/competitions/{COMPETITION_WYID}/seasons',       {}),
    ('/competitions',                                   {}),
    ('/seasons',                                        {'compId': COMPETITION_WYID}),
]
for path, params in candidates:
    resp = try_endpoint(path, **params)
    if resp is not None:
        print(f'    sample: {str(resp)[:200]}')
        print()